# 02. Rotations and inherited delay

How much of a flight's delay arrived with the aircraft? The answer needs the aircraft's day rebuilt
from tail numbers, and the propagation coefficient: the minutes of departure delay passed on per
minute the inbound flight was late beyond its scheduled turn less a minimum turn.

The dead end is kept here: keying rotations on the tail number and the date, the way the monthly
files are laid out. It splits every overnight.

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(3)
ROOT = Path(os.environ.get("TURNAROUND_NOTEBOOK_ROOT") or (Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent))
manifest = json.loads((ROOT / "results" / "manifest.json").read_text(encoding="utf-8"))
values, tables = manifest["values"], manifest["tables"]


def v(key):
    return values[key]["value"]


def t(key):
    # A manifest value formatted the way the documents print it.
    from turnaround_core.formats import format_value

    return format_value(values[key]["value"], values[key]["fmt"])


def table(key):
    entry = tables[key]
    return pl.DataFrame(entry["rows"], schema=entry["columns"], orient="row")


MARTS = ROOT / "results" / "marts"
print("manifest as of", manifest["as_of"], "with", len(values), "values;", "window", v("data.first_month"), "to", v("data.last_month"))

manifest as of 2026-10-01 with 625 values; window January 2015 to July 2026


## On the simulator, where the coefficient is known

Two simulated years: the minimum turn is chosen on the first, the coefficient and the inherited
share are reported on the second, and the truth is the simulator's own inherited minutes, capped
at each leg's departure delay as the estimator caps them.

In [2]:
from turnaround_chapters import ch04_inherited, padding
from turnaround_core.config import POLICY, TURN_BINS
from turnaround_rotations import reconstruct
from turnaround_sim.network import SimSpec, simulate

sim = simulate(SimSpec(seed=11, days=730, start="2022-01-01", meltdown_day=600))
con = duckdb.connect()
con.register("sim", sim.flights.to_arrow())
con.execute("create table raw as select * from sim")
padding.build_unimpeded(con, "raw", where_fit="year = 2022", percentile=0.10, min_flights=20)
con.execute(f"create table fct_flights as {padding.padded_flights_sql('raw')}")
con.execute(f"create table int_legs as {reconstruct.full_sql('fct_flights', POLICY.rotation_gap_hours)}")
counts = reconstruct.counts(con, "int_legs")
r = ch04_inherited.estimate(con, legs="int_legs", flights="fct_flights", fit_year=2022, test_first=2023, test_last=2023, bins=TURN_BINS)
truth_share = con.execute(
    "select sum(least(true_inherited, greatest(dep_delay, 0))) / sum(greatest(arr_delay, 0)) "
    "from fct_flights where year = 2023 and not cancelled and not diverted"
).fetchone()[0]
print(counts)
print(f"rho {r.estimate.rho:.3f} (truth {sim.truth.propagation}), minimum turn {r.estimate.min_turn} (truth {sim.truth.min_turn})")
print(f"inherited share {r.share:.3f} (interval {r.share_low:.3f} to {r.share_high:.3f}); truth {truth_share:.3f}")

RotationCounts(legs=330845, linked=227043, first=136, gap=88494, broken_chain=15101, impossible=71, rotations=103802, tails=136)
rho 0.814 (truth 0.8), minimum turn 35 (truth 35)
inherited share 0.255 (interval 0.254 to 0.257); truth 0.282


## Keying rotations on the date: a dead end, kept

The files give each flight a date, the local date of its scheduled departure, and grouping an
aircraft's legs by tail number and date is the first thing anyone writes. It cuts every chain at
midnight: the red eye that lands at dawn and turns forty minutes later, the island night that is
shorter than a long sit at a hub. Those links are exactly where a late arrival is passed to the
morning. The reconstruction instead orders each tail's legs in UTC and starts a new rotation only
after a sit longer than the stated gap.

Below, the same simulated legs with every link that crosses a local date cut, and what that
changes.

In [3]:
con.execute(
    "create table legs_by_date as "
    "select l.* replace (case when l.link_status = 'linked' and p.flight_date <> l.flight_date then 'gap' else l.link_status end as link_status) "
    "from int_legs l left join int_legs p on p.flight_id = l.prev_flight_id"
)
by_date = ch04_inherited.estimate(con, legs="legs_by_date", flights="fct_flights", fit_year=2022, test_first=2023, test_last=2023, bins=TURN_BINS)
cut, carried = ch04_inherited.across_dates(con, legs="int_legs", legs_where="year(flight_date) = 2023", rho=r.estimate.rho, min_turn=r.estimate.min_turn)
pl.DataFrame(
    [
        {"rule": "UTC order, new rotation after a long sit", "links in the reporting year": r.test_links,
         "links cut at midnight": 0, "inherited minutes lost": 0.0, "inherited share": r.share},
        {"rule": "tail number and local date", "links in the reporting year": by_date.test_links,
         "links cut at midnight": cut, "inherited minutes lost": carried, "inherited share": by_date.share},
    ]
)

rule,links in the reporting year,links cut at midnight,inherited minutes lost,inherited share
str,i64,i64,f64,f64
"""UTC order, new rotation after a long sit""",111833,0,0.000,0.255
"""tail number and local date""",111204,629,19.954,0.255


The simulator's nights are long: its last arrivals land late in the evening and its first
departures leave hours later, so the links the date rule cuts carry almost nothing and the share
barely moves. That is the simulator's limit, not a defence of the rule. The real network flies red
eyes into the morning banks and short nights in Hawaii, and the cell below reads what the date rule
would have cut there, counted by the chapters stage on the reporting years.

In [4]:
print("linked legs whose previous leg is on another local date:", v("ch4.across_dates"))
print("their share of all inherited minutes:", t("ch4.across_dates_share"))
print("rotations", v("ch4.rotations"), "from", v("ch4.legs"), "legs on", v("ch4.tails"), "tails;",
      "broken chains", v("ch4.broken_chain"), "impossible sequences", v("ch4.impossible"))
table("ch4.buffer_curve")

linked legs whose previous leg is on another local date: 751844
their share of all inherited minutes: 1.0%
rotations 18151842 from 72583211 legs on 10019 tails; broken chains 369126 impossible sequences 601945


Scheduled turn,Legs,Minutes passed on per late minute,Low,High
str,i64,f64,f64,f64
"""0 to 30 min""",223551,1.504,1.482,1.526
"""30 to 40 min""",1747421,1.093,1.088,1.097
"""40 to 50 min""",3901417,1.045,1.042,1.048
"""50 to 60 min""",3970697,1.038,1.035,1.042
"""60 to 75 min""",4338922,0.958,0.955,0.961
"""75 to 90 min""",1573575,0.811,0.806,0.817
"""90 to 120 min""",1111771,0.651,0.646,0.657
"""120 to 180 min""",712102,0.434,0.428,0.440
"""180 to 300 min""",428630,0.257,0.251,0.262


## The first flight of the aircraft's day

The traveler's version of the same mechanism: a first departure has no inbound flight to inherit
from. The leg position mart counts on time arrivals by the leg's place in the aircraft's local day.

In [5]:
legs = pl.read_parquet(MARTS / "mart_leg_position.parquet")
(
    legs.filter(pl.col("year") >= int(v("policy.test_first_year")))
    .group_by("leg_position")
    .agg(pl.col("flown").sum(), (pl.col("on_time").sum() / pl.col("flown").sum()).alias("on time rate"))
    .sort("leg_position")
    .head(8)
)

leg_position,flown,on time rate
i64,i64,f64
1,6125514,0.858
2,5652823,0.807
3,4813482,0.772
4,3809323,0.734
5,2289445,0.711
6,1912195,0.705
